# Advanced Temporal Pattern Charts (Solution)

Analyze seasonal/time patterns using calendar heatmap-like and lag plots.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

sns.set_theme(style="whitegrid")

def resolve_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "data" / "owid_co2_subset.csv").exists():
            return p
    raise FileNotFoundError("Cannot locate data/owid_co2_subset.csv")

root = resolve_repo_root()
df = pd.read_csv(root / "data" / "owid_co2_subset.csv")
df = df[df["iso_code"].astype(str).str.len() == 3].copy()
d_recent = df[df["year"] >= 2000].copy()
latest_year = (
    d_recent.dropna(subset=["co2_per_capita", "gdp", "population"])
    ["year"]
    .max()
)
d_latest = d_recent[d_recent["year"] == latest_year].copy()
d_latest.head()

## 1) Heatmap year x continent

In [ ]:
top_countries = (
    d_latest.groupby("country", as_index=False)["population"].mean()
    .nlargest(8, "population")["country"]
)
sub = d_recent[d_recent["country"].isin(top_countries)]
p = sub.pivot_table(index="country", columns="year", values="co2_per_capita", aggfunc="mean")

plt.figure(figsize=(13, 5))
sns.heatmap(p, cmap='mako', cbar_kws={'label':'CO2 per capita'})
plt.title('Temporal heatmap: CO2 per capita by country-year')
plt.tight_layout(); plt.show()

## 2) Seasonal subseries style (group by decade)

In [ ]:
d = d_recent.copy()
d['decade'] = (d['year'] // 10) * 10
s = d.groupby(['decade','country'],as_index=False)['co2_per_capita'].mean()
keep = s.groupby('country', as_index=False)['co2_per_capita'].mean().nlargest(6, 'co2_per_capita')['country']
s = s[s['country'].isin(keep)]
plt.figure(figsize=(10,4.5))
sns.lineplot(data=s, x='decade', y='co2_per_capita', hue='country', marker='o')
plt.title('Decade subseries: CO2 per capita (top emitters per capita)')
plt.tight_layout(); plt.show()

## 3) Lag plot

In [ ]:
target = 'United States'
us = d_recent[d_recent['country']==target].sort_values('year')[['year','co2_per_capita']].dropna()
us['lag1'] = us['co2_per_capita'].shift(1)

plt.figure(figsize=(5.5,5))
sns.scatterplot(data=us.dropna(), x='lag1', y='co2_per_capita')
plt.title(f'Lag plot ({target} CO2 per capita)')
plt.xlabel('lag-1')
plt.ylabel('current')
plt.tight_layout(); plt.show()

## Reflection
- Nêu 2 điểm học được về chart selection.
- Chỉ ra 1 rủi ro diễn giải sai với loại chart trong lab này.